# 🧠 Enhanced 2D CNN Gunshot Detector (Mel-Spectrogram)

> Uses **Mel-Spectrograms** instead of raw waveforms. More robust to speaker/mic distortion.
> 
> **Why 2D?** A Mel-Spectrogram captures the frequency content ("pitch") over time.
> Gunshots appear as bright vertical stripes (all frequencies firing simultaneously).
> This signature is preserved even when played through laptop speakers.

### Paper References
1. Salamon & Bello (2017) — Environmental Sound Classification with CNNs
2. Park et al. (2019) — SpecAugment (frequency & time masking)
3. Zhang et al. (2018) — MixUp augmentation
4. Magee et al. (2019) — 2D CNN ensemble on Raspberry Pi (IEEE)
5. Saha et al. (2025) — Spectrogram CNN for forest gunshot detection

In [ ]:
%pip install -q tensorflow librosa soundfile scikit-learn matplotlib seaborn tqdm pandas numpy

In [ ]:
import os, re, random, json, warnings
import numpy as np
import pandas as pd
import librosa
import librosa.display
import soundfile as sf
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm
from datetime import datetime

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from sklearn.model_selection import GroupShuffleSplit
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    fbeta_score
)

warnings.filterwarnings('ignore')
np.random.seed(42); tf.random.set_seed(42); random.seed(42)

print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {len(tf.config.list_physical_devices("GPU")) > 0}')
print('✅ Imports loaded.')

In [ ]:
# ============================================================
# CELL 3: CONFIGURATION
# ============================================================

DATA_DIR = Path(r'CHANGE_THIS_TO_YOUR_TRIMMED_DATA_PATH')

SAMPLE_RATE = 22050
CLIP_DURATION_MS = 750
TARGET_SAMPLES = int(SAMPLE_RATE * CLIP_DURATION_MS / 1000)

# --- Mel-Spectrogram Parameters ---
N_MELS = 64           # Number of Mel bands
N_FFT = 2048          # FFT window size
HOP_LENGTH = 512      # Hop between FFT windows
FMIN = 20             # Minimum frequency (Hz)
FMAX = 8000           # Maximum frequency (Hz)

# --- Training ---
BATCH_SIZE = 64
EPOCHS = 50
LEARNING_RATE = 0.001
TEST_SPLIT = 0.15
VAL_SPLIT = 0.15

# --- Augmentation ---
USE_AUGMENTATION = True
USE_MIXUP = True
MIXUP_ALPHA = 0.3
USE_FREQ_MASK = True   # SpecAugment frequency masking
USE_TIME_MASK = True   # SpecAugment time masking
USE_PITCH_SHIFT = True
USE_SPEED_PERTURB = True

OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Data: {DATA_DIR}')
print(f'Clip: {CLIP_DURATION_MS}ms | Mel: {N_MELS}x? | SR: {SAMPLE_RATE}')

In [ ]:
# ============================================================
# CELL 4: Augmentation Functions (applied on raw waveform BEFORE spectrogram)
# ============================================================

def augment_waveform(y, sr):
    aug = y.copy()
    if random.random() < 0.5:
        shift = int(len(aug) * random.uniform(-0.1, 0.1))
        aug = np.roll(aug, shift)
    if random.random() < 0.5:
        snr_db = random.uniform(15, 30)
        signal_power = np.mean(aug ** 2)
        noise_power = signal_power / (10 ** (snr_db / 10))
        aug = aug + np.random.normal(0, np.sqrt(max(noise_power, 1e-10)), len(aug)).astype(np.float32)
    if random.random() < 0.5:
        aug = aug * (10 ** (random.uniform(-6, 6) / 20))
    peak = np.max(np.abs(aug))
    if peak > 1e-6:
        aug = aug / peak
    return aug.astype(np.float32)

def pitch_shift_aug(y, sr, max_semitones=2):
    shifted = librosa.effects.pitch_shift(y=y, sr=sr, n_steps=random.uniform(-max_semitones, max_semitones))
    peak = np.max(np.abs(shifted))
    return (shifted / peak if peak > 1e-6 else shifted).astype(np.float32)

def speed_perturb_aug(y, target_len, max_rate=0.1):
    rate = 1.0 + random.uniform(-max_rate, max_rate)
    stretched = librosa.effects.time_stretch(y=y, rate=rate)
    if len(stretched) >= target_len:
        stretched = stretched[:target_len]
    else:
        stretched = np.pad(stretched, (0, target_len - len(stretched)))
    peak = np.max(np.abs(stretched))
    return (stretched / peak if peak > 1e-6 else stretched).astype(np.float32)

print('✅ Waveform augmentation functions defined.')

In [ ]:
# ============================================================
# CELL 5: Mel-Spectrogram Extraction + SpecAugment
# ============================================================
def waveform_to_mel(y, sr):
    """Convert raw waveform to log-Mel-spectrogram."""
    S = librosa.feature.melspectrogram(
        y=y, sr=sr, n_mels=N_MELS, n_fft=N_FFT,
        hop_length=HOP_LENGTH, fmin=FMIN, fmax=FMAX
    )
    S_dB = librosa.power_to_db(S, ref=np.max)
    S_norm = (S_dB + 80.0) / 80.0
    return np.clip(S_norm, 0.0, 1.0).astype(np.float32)
# SpecAugment on the spectrogram itself
# Ref: Park et al. (2019)
def specaugment_freq_mask(mel, max_bands=8):
    """Mask random frequency bands."""
    aug = mel.copy()
    n_bands = random.randint(1, max_bands)
    start = random.randint(0, mel.shape[0] - n_bands)
    aug[start:start + n_bands, :] = mel.min()
    return aug
def specaugment_time_mask(mel, max_frames=10):
    """Mask random time frames."""
    aug = mel.copy()
    n_frames = random.randint(1, max_frames)
    start = random.randint(0, mel.shape[1] - n_frames)
    aug[:, start:start + n_frames] = mel.min()
    return aug
# Compute expected mel shape
_test_mel = waveform_to_mel(np.zeros(TARGET_SAMPLES, dtype=np.float32), SAMPLE_RATE)
MEL_SHAPE = _test_mel.shape  # (N_MELS, time_frames)
print(f'Mel-Spectrogram shape: {MEL_SHAPE} → Model input: {MEL_SHAPE + (1,)}')


In [ ]:
# ============================================================
# CELL 6: Data Loading
# ============================================================

def extract_source_group(filename):
    name = Path(filename).stem
    parts = re.split(r'_onset\d+|_win\d+|_clip\d+|_\d{6}$', name)
    return parts[0] if parts else name

class1_dir = DATA_DIR / 'class_1_gunshot'
class0_dir = DATA_DIR / 'class_0_nongunshot'
assert class1_dir.exists() and class0_dir.exists()

files_1 = sorted(class1_dir.rglob('*.wav'))
files_0 = sorted(class0_dir.rglob('*.wav'))
all_files = files_1 + files_0
all_labels = [1] * len(files_1) + [0] * len(files_0)
all_groups = [extract_source_group(f.name) for f in all_files]

# Group-based split
gss_test = GroupShuffleSplit(n_splits=1, test_size=TEST_SPLIT, random_state=42)
remain_idx, test_idx = next(gss_test.split(all_files, all_labels, all_groups))
remain_files = [all_files[i] for i in remain_idx]
remain_labels = [all_labels[i] for i in remain_idx]
remain_groups = [all_groups[i] for i in remain_idx]
test_files = [all_files[i] for i in test_idx]
test_labels = [all_labels[i] for i in test_idx]

gss_val = GroupShuffleSplit(n_splits=1, test_size=VAL_SPLIT / (1 - TEST_SPLIT), random_state=42)
train_idx, val_idx = next(gss_val.split(remain_files, remain_labels, remain_groups))
train_files = [remain_files[i] for i in train_idx]
train_labels = [remain_labels[i] for i in train_idx]
val_files = [remain_files[i] for i in val_idx]
val_labels = [remain_labels[i] for i in val_idx]

print(f'Train: {len(train_files):,} | Val: {len(val_files):,} | Test: {len(test_files):,}')

In [ ]:
# ============================================================
# CELL 7: Load + Convert to Mel-Spectrograms
# ============================================================

def load_mel_dataset(file_list, label_list, augment=False):
    X, y = [], []
    for filepath, label in tqdm(zip(file_list, label_list), total=len(file_list), desc='Loading'):
        try:
            wav, _ = librosa.load(str(filepath), sr=SAMPLE_RATE, mono=True)
            wav = np.nan_to_num(wav)
            if len(wav) >= TARGET_SAMPLES:
                wav = wav[:TARGET_SAMPLES]
            else:
                wav = np.pad(wav, (0, TARGET_SAMPLES - len(wav)))
            peak = np.max(np.abs(wav))
            if peak > 1e-6:
                wav = wav / peak
        except Exception:
            continue
        
        mel = waveform_to_mel(wav, SAMPLE_RATE)
        X.append(mel)
        y.append(label)
        
        if augment and USE_AUGMENTATION:
            aug_wav = augment_waveform(wav, SAMPLE_RATE)
            if USE_PITCH_SHIFT and random.random() < 0.3:
                aug_wav = pitch_shift_aug(aug_wav, SAMPLE_RATE)
            if USE_SPEED_PERTURB and random.random() < 0.3:
                aug_wav = speed_perturb_aug(aug_wav, TARGET_SAMPLES)
            aug_mel = waveform_to_mel(aug_wav, SAMPLE_RATE)
            if USE_FREQ_MASK and random.random() < 0.4:
                aug_mel = specaugment_freq_mask(aug_mel)
            if USE_TIME_MASK and random.random() < 0.4:
                aug_mel = specaugment_time_mask(aug_mel)
            X.append(aug_mel)
            y.append(label)
    
    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.float32)
    X = X[..., np.newaxis]  # Add channel dim: (N, n_mels, time, 1)
    return X, y

print('📥 Loading TRAINING...')
X_train, y_train = load_mel_dataset(train_files, train_labels, augment=True)
print('📥 Loading VALIDATION...')
X_val, y_val = load_mel_dataset(val_files, val_labels, augment=False)
print('📥 Loading TEST...')
X_test, y_test = load_mel_dataset(test_files, test_labels, augment=False)

print(f'\nX_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}')

In [ ]:
def mixup_generator(X, y, batch_size, class_weights, alpha=MIXUP_ALPHA):
    indices = np.arange(len(X))
    cw0 = class_weights[0]
    cw1 = class_weights[1]
    while True:
        np.random.shuffle(indices)
        for i in range(0, len(X), batch_size):
            batch_idx = indices[i:i+batch_size]
            X_batch = X[batch_idx].copy()
            y_batch = y[batch_idx].copy()
            if USE_MIXUP:
                lam = np.random.beta(alpha, alpha, size=len(X_batch))
                lam_x = lam.reshape(-1, 1, 1, 1)
                mix_idx = np.random.permutation(len(X_batch))
                X_batch = lam_x * X_batch + (1 - lam_x) * X_batch[mix_idx]
                y_batch = lam * y_batch + (1 - lam) * y_batch[mix_idx]
            s_weights = y_batch * cw1 + (1 - y_batch) * cw0
            yield (
                X_batch.astype(np.float32),
                (y_batch.astype(np.float32), y_batch.astype(np.float32)),
                (s_weights.astype(np.float32), np.ones_like(y_batch, dtype=np.float32))
            )
print('MixUp generator defined.')


In [ ]:
# ============================================================
# CELL 9: DUAL-HEAD 2D CNN Architecture
# ============================================================

def build_enhanced_2d_cnn(input_shape):
    inputs = layers.Input(shape=input_shape, name='mel_input')
    
    # --- Shared Backbone ---
    x = layers.Conv2D(16, (3, 3), activation='relu', padding='same', name='conv1')(inputs)
    x = layers.BatchNormalization(name='bn1')(x)
    x = layers.MaxPooling2D((2, 2), name='pool1')(x)
    
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same', name='conv2')(x)
    x = layers.BatchNormalization(name='bn2')(x)
    x = layers.MaxPooling2D((2, 2), name='pool2')(x)
    
    x = layers.Conv2D(64, (3, 3), activation='relu', padding='same', name='conv3')(x)
    x = layers.BatchNormalization(name='bn3')(x)
    x = layers.MaxPooling2D((2, 2), name='pool3')(x)
    
    x = layers.Conv2D(128, (3, 3), activation='relu', padding='same', name='conv4')(x)
    x = layers.BatchNormalization(name='bn4')(x)
    
    shared = layers.GlobalAveragePooling2D(name='gap')(x)
    
    # --- Head 1: Gunshot ---
    h1 = layers.Dense(64, activation='relu', name='gunshot_dense')(shared)
    h1 = layers.Dropout(0.4, name='gunshot_dropout')(h1)
    gunshot_out = layers.Dense(1, activation='sigmoid', name='gunshot_output')(h1)
    
    # --- Head 2: Anomaly ---
    h2 = layers.Dense(32, activation='relu', name='anomaly_dense')(shared)
    h2 = layers.Dropout(0.3, name='anomaly_dropout')(h2)
    anomaly_out = layers.Dense(1, activation='sigmoid', name='anomaly_output')(h2)
    
    model = models.Model(inputs=inputs, outputs={'gunshot_output': gunshot_out, 'anomaly_output': anomaly_out},
                         name='Enhanced_2D_CNN_DualHead')
    return model

model = build_enhanced_2d_cnn(X_train.shape[1:])

class_weights_arr = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = {i: w for i, w in enumerate(class_weights_arr)}

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss={'gunshot_output': 'binary_crossentropy', 'anomaly_output': 'binary_crossentropy'},
    loss_weights={'gunshot_output': 1.0, 'anomaly_output': 0.3},
    metrics={'gunshot_output': ['accuracy', keras.metrics.Precision(name='precision'),
                                keras.metrics.Recall(name='recall'), keras.metrics.AUC(name='auc')],
             'anomaly_output': ['accuracy']}
)

model.summary()
print(f'\n📊 Total parameters: {model.count_params():,}')

In [ ]:
# ============================================================
# CELL 10: Training
# ============================================================
train_gen = mixup_generator(X_train, y_train, BATCH_SIZE, class_weight_dict)
steps_per_epoch = int(np.ceil(len(X_train) / BATCH_SIZE))
history = model.fit(
    train_gen,
    steps_per_epoch=steps_per_epoch,
    validation_data=(X_val, (y_val, y_val.copy())),
    epochs=EPOCHS,
    callbacks=[
        callbacks.EarlyStopping(monitor='val_gunshot_output_auc', patience=8, mode='max', restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor='val_gunshot_output_loss', factor=0.5, patience=4, min_lr=1e-6, mode='min'),
        callbacks.ModelCheckpoint(str(OUTPUT_DIR / 'enhanced_2d_cnn_best.h5'), monitor='val_gunshot_output_auc', mode='max', save_best_only=True),
    ],
    verbose=1
)
print('\nTraining complete!')


In [ ]:
# ============================================================
# CELL 11: Evaluation
# ============================================================

preds = model.predict(X_test, verbose=0)
y_pred_gunshot = preds['gunshot_output'] if isinstance(preds, dict) else preds[0]
y_pred_anomaly = preds['anomaly_output'] if isinstance(preds, dict) else preds[1]
y_pred_binary = (y_pred_gunshot.flatten() >= 0.5).astype(int)

print('\n📊 Classification Report:')
print(classification_report(y_test, y_pred_binary, target_names=['Background', 'Gunshot']))

cm = confusion_matrix(y_test, y_pred_binary)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Background', 'Gunshot'], yticklabels=['Background', 'Gunshot'])
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.title('Enhanced 2D CNN — Confusion Matrix')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'enhanced_2d_cnn_confusion.png', dpi=150)
plt.show()

f2 = fbeta_score(y_test, y_pred_binary, beta=2)
roc = roc_auc_score(y_test, y_pred_gunshot.flatten())
print(f'🎯 F2 Score: {f2:.4f} | ROC-AUC: {roc:.4f}')

results = {'f2_score': float(f2), 'roc_auc': float(roc), 'clip_duration_ms': CLIP_DURATION_MS,
           'augmentations': {'mixup': USE_MIXUP, 'specaugment_freq': USE_FREQ_MASK,
                             'specaugment_time': USE_TIME_MASK, 'pitch_shift': USE_PITCH_SHIFT,
                             'speed_perturb': USE_SPEED_PERTURB},
           'mel_params': {'n_mels': N_MELS, 'n_fft': N_FFT, 'hop_length': HOP_LENGTH},
           'timestamp': datetime.now().isoformat()}
with open(OUTPUT_DIR / 'enhanced_2d_cnn_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f'💾 Saved to {OUTPUT_DIR}')